In [38]:
import numpy as np
import math
import plotly.graph_objects as go

# — link lengths (mm) —
a1, a2, a3 = 37.0, 63.54, 200.0
Dmax = a1 + a2 + a3

# — raw‐angle limits (°) —
hip_lim   = (-46.0,  46.0)
knee_lim  = (-91.0,  91.0)
ankle_lim = (-121.0,   0.0)   # raw θ₃ must be ≤ 0 (down is negative)

# — sample hip yaw over its range —
theta_hip = np.linspace(math.radians(hip_lim[0]),
                       math.radians(hip_lim[1]), 180)

# — heights to sweep —
z_values = np.linspace(0, Dmax, 51)

def ik_elbow_in(x, y, z):
    # 1) overall reach
    if math.hypot(x, y, z) > Dmax:
        return False

    # 2) hip yaw
    th1 = math.degrees(math.atan2(y, x))

    # 3) project into leg plane
    r1 = math.hypot(x, y) - a1
    phi2 = math.atan2(z, r1)
    r3   = math.hypot(r1, z)

    # 4) knee (elbow-in)
    c1 = (a3*a3 - a2*a2 - r3*r3) / (-2 * a2 * r3)
    c1 = max(-1, min(1, c1))
    phi1 = math.acos(c1)
    th2  = math.degrees(phi2 - phi1)

    # 5) ankle (same inversion as elbow-out)
    c3 = (r3*r3 - a2*a2 - a3*a3) / (-2 * a2 * a3)
    c3 = max(-1, min(1, c3))
    phi3 = math.acos(c3)
    th3 = -math.degrees(math.pi - phi3)

    # 6) raw‐limit check
    return (hip_lim[0] <= th1 <= hip_lim[1] and
            knee_lim[0] <= th2 <= knee_lim[1] and
            ankle_lim[0] <= th3 <= ankle_lim[1])

# — build frames —
frames = []
for z0 in z_values:
    xs, ys = [], []
    for th in theta_hip:
        lo, hi = 0.0, Dmax
        for _ in range(20):
            mid = 0.5*(lo + hi)
            x = mid * math.cos(th)
            y = mid * math.sin(th)
            if ik_elbow_in(x, y, z0):
                lo = mid
            else:
                hi = mid
        xs.append(lo * math.cos(th))
        ys.append(lo * math.sin(th))

    frames.append(go.Frame(
        data=[go.Scatter(x=xs, y=ys, fill='toself',
                         hovertemplate="x: %{x:.1f} mm<br>y: %{y:.1f} mm<br>z: "+f"{z0:.1f} mm<extra></extra>")],
        name=f"{z0:.1f}"
    ))

# — initial figure & slider —
fig = go.Figure(data=[frames[0].data[0]], frames=frames)
steps = [
    {"label":f"{z0:.1f}", "method":"animate",
     "args":[[f"{z0:.1f}"],{"mode":"immediate","frame":{"duration":0},"transition":{"duration":0}}]}
    for z0 in z_values
]
fig.update_layout(
    title="Elbow-In Workspace vs. z₀",
    width=800, height=800,
    xaxis=dict(title="X (mm)", scaleanchor="y", scaleratio=1),
    updatemenus=[{
        "type":"buttons",
        "buttons":[{"label":"Play","method":"animate",
                    "args":[None,{"frame":{"duration":100,"redraw":True},"fromcurrent":True}]}]
    }],
    sliders=[{"active":0,"currentvalue":{"prefix":"z₀ (mm): "},"steps":steps}],
    margin=dict(l=50,r=50,t=80,b=50)
)
fig.show()
